# RL Day 2 — RR 對標版 Colab 教材
**Heli → Fighter**

> 上課前請先點：**File → Save a copy in Drive**，建立自己的副本。

今天會看到比較即時、連續狀態的遊戲。請先觀察動態，再調整 **🔧** 參數。

In [ ]:
# ── Setup：下載 RR Colab 環境並匯入 ─────────────────────────
RR_ENVS_URL = 'https://raw.githubusercontent.com/colombo0718/rl-lab-group-b/master/rr_envs.py'

!rm -f rr_envs.py
!wget -q "{RR_ENVS_URL}" -O rr_envs.py
!pip install gymnasium matplotlib numpy -q

import importlib, sys, time
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, clear_output, display

if 'rr_envs' in sys.modules:
    del sys.modules['rr_envs']

from rr_envs import (
    MABEnv, Maze1DEnv, Maze2DEnv, HeliEnv, FighterEnv,
    show_env, animate_actions, animate_random, run_q_learning, plot_training, plot_maze2d_qtable,
)

def make_key_fn(env, bins=None):
    from gymnasium import spaces
    if isinstance(env.observation_space, spaces.Discrete):
        return lambda obs: int(obs)
    obs_dim = env.observation_space.shape[0]
    if bins is None:
        bins = 6
    low, high = env.observation_space.low, env.observation_space.high
    edges = [np.linspace(low[i], high[i], bins + 1) for i in range(obs_dim)]
    def to_key(obs):
        return tuple(int(np.clip(np.digitize(obs[i], edges[i]) - 1, 0, bins - 1)) for i in range(obs_dim))
    return to_key

def animate_policy(env, Q, steps=40, bins=None, delay=0.25):
    to_key = make_key_fn(env, bins=bins)
    obs, info = env.reset()
    for step in range(steps):
        clear_output(wait=True)
        display(HTML(env.render_html()))
        state = to_key(obs)
        action = max(range(env.action_space.n), key=lambda a: Q.get((state, a), 0.0))
        obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        print(f'step={step+1}  action={action}  reward={reward:.2f}  done={done}')
        time.sleep(delay)
        if done:
            clear_output(wait=True)
            display(HTML(env.render_html()))
            print(f'finished at step={step+1}, final reward={reward:.2f}')
            break

print('✅ RR Colab environments ready!')

---
## Part 1 · Heli 直升機

這個環境對標 RR 平台 `heli.html`。

- 存活每步：`+0.01`
- 通過牆：`+1`
- 撞牆 / 撞邊界：`-10`

動作：`0 = fall`，`1 = flap`。

In [ ]:
# ═══════════════ 🔧 只改這裡 ═══════════════
HELI_MODE = 'fixed'      # 'fixed' / 'small' / 'large'
EPISODES = 600
ALPHA = 0.5
GAMMA = 0.95
EPSILON = 0.25
BINS = 6
# ═══════════════════════════════════════════

env = HeliEnv(mode=HELI_MODE)
env.reset()
show_env(env)

In [ ]:
# 先看隨機動作下的遊戲畫面
env = HeliEnv(mode=HELI_MODE)
animate_random(env, steps=80, delay=0.06, stop_on_done=True)

In [ ]:
env = HeliEnv(mode=HELI_MODE)
rewards, lengths, Q = run_q_learning(
    env, alpha=ALPHA, gamma=GAMMA, epsilon=EPSILON, n_episodes=EPISODES, bins=BINS, verbose=False
)
plot_training(rewards, lengths, label=f'Heli mode={HELI_MODE}, epsilon={EPSILON}', window=30)

print('訓練後的 greedy policy：')
env = HeliEnv(mode=HELI_MODE)
animate_policy(env, Q, steps=120, bins=BINS, delay=0.06)

---
## Part 2 · Fighter 戰機挑戰

這個環境對標 RR 平台 `fighter.html`。

- 子彈擊中岩石：`+10`
- 子彈飛出畫面 miss：`-1`
- 岩石撞到玩家：`-100`
- 擊中 10 次：clear

動作：`0 = none`，`1 = left`，`2 = right`，`3 = shoot`。

In [ ]:
# ═══════════════ 🔧 只改這裡 ═══════════════
FIGHTER_MODE = 'fixed'   # 'fixed' / 'randomX' / 'randomXY' / 'falling' / 'drifting'
EPISODES = 800
ALPHA = 0.5
GAMMA = 0.95
EPSILON = 0.3
BINS = 6
# ═══════════════════════════════════════════

env = FighterEnv(mode=FIGHTER_MODE)
env.reset()
show_env(env)

In [ ]:
# 先看固定策略：射擊後等待
env = FighterEnv(mode=FIGHTER_MODE)
env.reset()
animate_actions(env, [3] + [0] * 30, delay=0.08, stop_on_done=True)

In [ ]:
env = FighterEnv(mode=FIGHTER_MODE)
rewards, lengths, Q = run_q_learning(
    env, alpha=ALPHA, gamma=GAMMA, epsilon=EPSILON, n_episodes=EPISODES, bins=BINS, verbose=False
)
plot_training(rewards, lengths, label=f'Fighter mode={FIGHTER_MODE}, epsilon={EPSILON}', window=30)

print('訓練後的 greedy policy：')
env = FighterEnv(mode=FIGHTER_MODE)
animate_policy(env, Q, steps=80, bins=BINS, delay=0.08)